# Value-Based Reinforcement Learning

$U_t = R_t + \gamma R_{t+1} + \gamma ^ 2 R_{t+2} + \gamma ^ 3 R_{t+3} + ...$
- `policy function`: $\mathbb{P} [A=a | S=s] = \pi(a|s)$.
- `state transition`: $\mathbb{P} [S^{'} = s^{'} | S=s, A=a] = p(s^{'}|s, a)$.

动作价值函数：$Q_\pi(s_t, a_t) = \mathbb E [U_t | S_t = s_t, A_t = a_t]$.

最优动作价值函数：$Q^*_\pi(s_t, a_t) = max_\pi Q_\pi(s_t, a_t)$（不管采用何种策略函数$\pi$，采取动作$a_t$的收益最高，函数与$\pi$无关）

## Deep Q-Network(DQN)

> 采用神经网络近似$Q*$函数。
>
> 强化学习**目标**：在游戏结束时，获得的奖励总和越大越好。
>
> **问题**：如果$Q^*$函数是已知的，最好的行为`action`是什么？
>
> $a^* = argmax_a Q^*(s, a)$，$Q^*$作为一个“先知”，它能告诉你每个动作带来的平均回报，你应该听先知的话，采取平均回报最高的动作。
>
> 但目前问题是我们并没有这样的“先知”函数。
>
> `Deep Q Network`即采用神经网络近似$Q^*$函数，记神经网络$Q(s, a; w)$。

state $s_t$ ---Conv---> feature ---Dense---> ($Q(s_t, left;, w_t), Q(s_t, right;, w_t), Q(s_t, up;, w_t)$)

- 当前观测到状态$s_t$，采用`DQN`将$s_t$作为输入，给所有动作打分：$a_t = argmax_a Q(s_t, a; w)$，选出分数最高的动作作为$a_t$；
- `agent`执行$a_t$动作后，环境会改变状态，使用状态转移函数来抽取新的状态：$s_{t+1} \sim p(\cdot | s_t, a_t)$；
- 环境同时告诉我们这一步的$r_t$，即强化学习中的监督信息，`DQN`通过$r_t$训练；
- 有了新的状态$s_{t+1}$，`DQN`再对所有动作打分……

## Temporal Difference(TD) Learning

eg：假如开车从NYC到Atlanta，预估用时。
- 做出一个预测：$q=Q(w)$，假设为1000分钟；
- 完成一次旅行得到目标值$y$，假设为860分钟；
- 计算损失：$L = \frac{1}{(q-y)^2}$；
- 计算梯度：$\frac {\partial L}{\partial w} = \frac {\partial q}{\partial w} \cdot \frac {\partial L}{\partial q} = (q - y) \cdot \frac {\partial Q(w)}{\partial w}$；
- 梯度下降：$w_{t+1} = w_t - \alpha \cdot \frac {\partial L}{\partial w} | w = w_t$.

提问：如果在不完成一次旅行的情况下，是否可以进行模型参数的更新？

比如在从NYC开车到Atlanta的路上，在抵达DC时汽车抛锚，无法完成后续的旅行，已知从NYC到DC花费300分钟。
- 模型预测：$Q(x) = 1000$；
- 更新模型预测：模型预测从DC到Atlanta用时600分钟：$TD \ target = 300 + 600 = 900$；
- `TD target`比最初的1000分钟更为可靠。

在DC时，即可将`TD target`当作$y$，对模型进行参数更新。

## How to apply TD learning to DQN?

在上面“驾驶时间”例子中，我们得到了下列公式：
$$
T_{NYC \rightarrow ATL} \approx T_{NYC \rightarrow DC} + T_{DC \rightarrow ATL}
$$
等式左边一项，右边有两项，其中有一项是真实观测到的。

在深度强化学习中：
$$
Q(s_t, a_t; w) \approx r_t + \gamma \cdot Q(s_{t+1}, a_{t+1}; w)
$$
- 等式左边为`t`时刻做的估计，这是未来奖励总和的期望；
- $r_t$为真实观测到的奖励；
- 另外一项为`DQN`在`t+1`时刻做的估计。

再看折扣回报：
$$
\begin{align}
U_t = R_t + \gamma R_{t+1} + \gamma ^ 2 R_{t+2} + \gamma ^ 3 R_{t+3} + \gamma ^ 4 R_{t+4} + ... \\
= R_t + \gamma (R_{t+1} + \gamma R_{t+2} + \gamma ^ 2 R_{t+3} + \gamma ^ 3 R_{t+4} + ... ) \\
= R_T + \gamma U_{t+1}
\end{align}
$$

- `DQN`的输出：$Q(s_t, a_t; w)$是对平均折扣回报的估计（$\mathbb E[U_t]$）
- `DQN`的输出：$Q(s_{t+1}, a_{t+1}; w)$是对平均折扣回报的估计（$\mathbb E[U_{t+1}]$）

于是：
$$
Q(s_t, a_t; w) \approx \mathbb E[R_t + \gamma \cdot Q(S_{t+1}, A_{t+1}; w)]
$$
- $Q(s_t, a_t; w) \approx \mathbb E[U_t]$；
- $Q(S_{t+1}, A_{t+1}; w) \approx \mathbb E[U_{t+1}]$；
- 等式左边为`Prediction`，右边为`TD Target`。

`Prediction`：$Q(s_t, a_t; w_t)$.

`TD target`：
$$
\begin{align}
y_t = r_t + \gamma \cdot Q(s_{t+1}, a_{a+1}; w_t) \\
= r_t + \gamma \cdot max_a Q(s_{t+1}, a; w_t).
\end{align}
$$

Loss：
$$
L_t = \frac 1 2 [Q(s_t, a_t; w) - y_t]^2.
$$

梯度下降：
$$
w_{t+1} = w_t - \alpha \cdot \frac {\partial L_t}{\partial w} | w=w_t.
$$